In [1]:
import pandas as pd
import numpy as np
import os
import sys
import requests
from tqdm import tqdm
import shutil
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from transformers import CLIPVisionModel

from google.colab import drive

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
project_root = '/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it'

if project_root not in sys.path:
    sys.path.append(project_root)

# Import code for segmentation / artifact detection
from src.model.discriminator.artifact_detector.ad_inference import process_images

In [22]:
# Paths
# Dataset Path
REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it'
DATASET_INVENTORY_PATH = f'{REPO_DIR}/datasets/dataset_inventory.csv'
INPUT_IMG_DIR = f'{REPO_DIR}/datasets/input/'

# Checkpoint Save Path
DATASET_CHECKPOINT_PATH = f'{REPO_DIR}/datasets/dataset_inventory_checkpoint.csv'
FAKE_DATASET_CHECKPOINT_PATH = f'{REPO_DIR}/datasets/fake_dataset_inventory_checkpoint.csv'

# CLIP Model Path
MODEL_PATH = "/content/drive/MyDrive/Model/clip_model"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = "/content/drive/MyDrive/Model/clip_classification_v6"

# AD Path
repo_path = Path(REPO_DIR)
adinf_dir = repo_path / 'src/model/discriminator/artifact_detector'
cfg_weights = adinf_dir / 'checkpoints/ad_richhf_baseline_model.bin'
cfg_heatmap_dir = repo_path / 'datasets/output/ad/heatmap'
cfg_mask_dir = repo_path / 'datasets/output/ad/mask'
cfg_mask_txt_dir = repo_path / 'datasets/output/ad/mask_txt'
input_img_path = repo_path / 'datasets/input/ai'

# **1. Setup**

In [5]:
# Load the inference image dataset
dataset_inventory = pd.read_csv(DATASET_INVENTORY_PATH)
dataset_inventory = dataset_inventory.copy()

DATASETS = [
    'imagenet_ai_0424_sdv5',
    'imagenet_ai_0419_vqdm',
    'imagenet_ai_0508_adm',
    'imagenet_ai_0419_biggan',
    'imagenet_glide',
]

dataset_inventory = dataset_inventory[(dataset_inventory['dataset'].isin(DATASETS)) & (dataset_inventory['split'] == 'test')]
dataset_inventory = dataset_inventory[['file_path', 'file_name', 'dataset', 'label', 'is_fake']]
dataset_inventory.head()

,file_path,file_name,dataset,label,is_fake
373,/content/drive/MyDrive/TrainingData/imagenet_a...,878_sdv5_00027.png,imagenet_ai_0424_sdv5,ai,1
415,/content/drive/MyDrive/TrainingData/imagenet_a...,882_sdv5_00039.png,imagenet_ai_0424_sdv5,ai,1
572,/content/drive/MyDrive/TrainingData/imagenet_a...,903_sdv5_00009.png,imagenet_ai_0424_sdv5,ai,1
613,/content/drive/MyDrive/TrainingData/imagenet_a...,907_sdv5_00020.png,imagenet_ai_0424_sdv5,ai,1
699,/content/drive/MyDrive/TrainingData/imagenet_a...,918_sdv5_00039.png,imagenet_ai_0424_sdv5,ai,1


In [6]:
# Download the image dataset
os.makedirs(INPUT_IMG_DIR, exist_ok=True)

print(f"Copying {len(dataset_inventory)} images...")

new_paths = []

for index, row in tqdm(dataset_inventory.iterrows(), total=dataset_inventory.shape[0]):
    source_path = row['file_path']
    dest_path = os.path.join(INPUT_IMG_DIR, row['file_name'])

    # 2. Append the new destination path to our list regardless of whether we skip copying
    new_paths.append(dest_path)

    # Skip if file already exists in the destination
    if not os.path.exists(dest_path):
        try:
            shutil.copy2(source_path, dest_path)
        except Exception as e:
            print(f"Error copying {row['file_name']}: {e}")

# 3. Overwrite the original column with our new list of local paths
dataset_inventory['new_file_path'] = new_paths

print("Copy process complete! DataFrame paths updated.")
dataset_inventory.head()

Copying 2000 images...


  0%|          | 0/2000 [00:00<?, ?it/s]

Copy process complete! DataFrame paths updated.


,file_path,file_name,dataset,label,is_fake,new_file_path
373,/content/drive/MyDrive/TrainingData/imagenet_a...,878_sdv5_00027.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...
415,/content/drive/MyDrive/TrainingData/imagenet_a...,882_sdv5_00039.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...
572,/content/drive/MyDrive/TrainingData/imagenet_a...,903_sdv5_00009.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...
613,/content/drive/MyDrive/TrainingData/imagenet_a...,907_sdv5_00020.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...
699,/content/drive/MyDrive/TrainingData/imagenet_a...,918_sdv5_00039.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...


In [7]:
# Load Discriminator
class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        return self.classifier(outputs.pooler_output)

model = ClassificationCLIP(MODEL_PATH).to(DEVICE)
model.load_state_dict(torch.load(
    os.path.join(SAVE_DIR, 'best_model.pth'),
    map_location=DEVICE, weights_only=False
))
model.eval()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.o

ClassificationCLIP(
  (vision_encoder): CLIPVisionModel(
    (vision_model): CLIPVisionTransformer(
      (embeddings): CLIPVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (position_embedding): Embedding(257, 1024)
      )
      (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-23): 24 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActiv

# **2. Classification Layer**

### 1a. CLIP Model

In [8]:
clip_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])

In [9]:
class InferenceDataset(Dataset):
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = float(1 - row['is_fake'])
        try:
            img = Image.open(row['file_path']).convert('RGB')
            img = self.transform(img)
        except Exception:
            img = torch.zeros(3, 224, 224)
        return img, torch.tensor([label], dtype=torch.float32), row['dataset'], int(row['is_fake'])

BATCH_SIZE = 128
eval_ds = InferenceDataset(dataset_inventory, clip_transform)
eval_dl = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=4, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [10]:
results = defaultdict(lambda: {'correct': 0, 'total': 0,
                                'real_correct': 0, 'real_total': 0,
                                'fake_correct': 0, 'fake_total': 0})

# 1. Initialize the list to store fake probabilities
all_prob_fake = []

print("Starting evaluation...")
with torch.no_grad():
    for imgs, labels, datasets, is_fakes in tqdm(eval_dl, desc="Evaluating"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(imgs)

        preds   = (torch.sigmoid(logits) >= 0.5).float()
        correct = (preds == labels)

        # 2. Calculate probability of being FAKE (1 - P(Real))
        # .squeeze(1) handles the [batch_size, 1] shape
        prob_fake = (1 - torch.sigmoid(logits)).squeeze(1).cpu().float().numpy()
        all_prob_fake.extend(prob_fake.tolist())

        for i in range(len(labels)):
            ds      = datasets[i]
            is_fake = is_fakes[i].item()
            ok      = correct[i].item()

            results[ds]['total']   += 1
            results[ds]['correct'] += ok

            if is_fake:
                results[ds]['fake_total']   += 1
                results[ds]['fake_correct'] += ok
            else:
                results[ds]['real_total']   += 1
                results[ds]['real_correct'] += ok

Starting evaluation...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

In [11]:
print("\n" + "=" * 65)
print(f"{'Dataset':<30} {'Overall':>8} {'Real':>8} {'Fake':>8}")
print("-" * 65)
total_correct = total_all = 0
DATASETS = dataset_inventory['dataset'].unique()
for ds in DATASETS:
    r = results[ds]
    overall = r['correct'] / r['total'] * 100 if r['total'] else 0
    real    = r['real_correct'] / r['real_total'] * 100 if r['real_total'] else 0
    fake    = r['fake_correct'] / r['fake_total'] * 100 if r['fake_total'] else 0
    total_correct += r['correct']
    total_all     += r['total']
    print(f"{ds:<30} {overall:>7.2f}%  {real:>7.2f}%  {fake:>7.2f}%")
print("-" * 65)
print(f"{'TOTAL':<30} {total_correct/total_all*100:>7.2f}%")
print("=" * 65)


Dataset                         Overall     Real     Fake
-----------------------------------------------------------------
imagenet_ai_0424_sdv5            94.75%    97.00%    92.50%
imagenet_ai_0419_vqdm            97.25%    98.00%    96.50%
imagenet_ai_0508_adm             99.25%    99.00%    99.50%
imagenet_ai_0419_biggan          99.25%    98.50%   100.00%
imagenet_glide                   98.50%    97.00%   100.00%
-----------------------------------------------------------------
TOTAL                            97.80%


In [12]:
dataset_inventory['original_CLIP_pred_label'] = [1 if p >= 0.5 else 0 for p in all_prob_fake]
dataset_inventory['original_CLIP_pred_confident'] = [round(p, 4) for p in all_prob_fake]

print("\nPredictions added to dataset_inventory successfully!")
dataset_inventory.head()


Predictions added to dataset_inventory successfully!


,file_path,file_name,dataset,label,is_fake,new_file_path,original_CLIP_pred_label,original_CLIP_pred_confident
373,/content/drive/MyDrive/TrainingData/imagenet_a...,878_sdv5_00027.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,0.9932
415,/content/drive/MyDrive/TrainingData/imagenet_a...,882_sdv5_00039.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,0.5303
572,/content/drive/MyDrive/TrainingData/imagenet_a...,903_sdv5_00009.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,0.9575
613,/content/drive/MyDrive/TrainingData/imagenet_a...,907_sdv5_00020.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,1.0000
699,/content/drive/MyDrive/TrainingData/imagenet_a...,918_sdv5_00039.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,1.0000


In [13]:
# Migrate the Image
def move_image_and_update_path(row):
    current_full_path = row['new_file_path']
    label = row['original_CLIP_pred_label']

    directory = os.path.dirname(current_full_path)
    filename = os.path.basename(current_full_path)

    if label == 0:
        subfolder = 'real'
    elif label == 1:
        subfolder = 'ai'
    else:
        return current_full_path

    new_dir_path = os.path.join(directory, subfolder)
    os.makedirs(new_dir_path, exist_ok=True)

    new_full_path = os.path.join(new_dir_path, filename)

    # Move the file
    if os.path.exists(current_full_path):
        try:
            shutil.move(current_full_path, new_full_path)
            return new_full_path
        except Exception as e:
            print(f"Error moving file {filename}: {e}")
            return current_full_path
    else:
        print(f"File not found, skipping: {current_full_path}")
        return current_full_path

dataset_inventory['new_file_path'] = dataset_inventory.apply(move_image_and_update_path, axis=1)

print("File migration and path updates complete!")

File migration and path updates complete!


# 2. Segmentation, Extract Binary Mask

In [18]:
# Filter to only those which are classified as Fake Image
dataset_inventory_fake = dataset_inventory[dataset_inventory['original_CLIP_pred_label'] == 1]
dataset_inventory_fake.head()

,file_path,file_name,dataset,label,is_fake,new_file_path,original_CLIP_pred_label,original_CLIP_pred_confident
373,/content/drive/MyDrive/TrainingData/imagenet_a...,878_sdv5_00027.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,0.9932
415,/content/drive/MyDrive/TrainingData/imagenet_a...,882_sdv5_00039.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,0.5303
572,/content/drive/MyDrive/TrainingData/imagenet_a...,903_sdv5_00009.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,0.9575
613,/content/drive/MyDrive/TrainingData/imagenet_a...,907_sdv5_00020.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,1.0000
699,/content/drive/MyDrive/TrainingData/imagenet_a...,918_sdv5_00039.png,imagenet_ai_0424_sdv5,ai,1,/content/drive/MyDrive/Colab Notebooks/deepfak...,1,1.0000


In [19]:
dataset_inventory_fake.shape

(998, 8)

In [20]:
process_images(
    input_dir=input_img_path,
    output_heatmap_dir=cfg_heatmap_dir,
    output_mask_dir=cfg_mask_dir,
    output_mask_txt_dir=cfg_mask_txt_dir,
    model_weights_path=cfg_weights,
    device=DEVICE,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


Loading weights:   0%|          | 0/1156 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b5
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

In [21]:
def add_segmentation_paths(df):
    """
    Adds segmentation output paths for rows where the image is classified as AI fake.
    """
    # initialize the new columns with NaN (Not a Number/Blank)
    df['CLIP_heatmap_dir'] = np.nan
    df['CLIP_mask_dir'] = np.nan
    df['CLIP_masktxt_dir'] = np.nan

    # create a mask to locate only the rows where the image is an AI fake
    is_fake = df['original_CLIP_pred_label'] == 1

    # extract the filename
    file_stems = df.loc[is_fake, 'file_name'].apply(lambda x: Path(x).stem)

    # construct the new paths
    df.loc[is_fake, 'CLIP_heatmap_dir'] = file_stems.apply(
        lambda stem: str(cfg_heatmap_dir / f"result_{stem}.png")
    )
    df.loc[is_fake, 'CLIP_mask_dir'] = file_stems.apply(
        lambda stem: str(cfg_mask_dir / f"mask_{stem}.png")
    )
    df.loc[is_fake, 'CLIP_masktxt_dir'] = file_stems.apply(
        lambda stem: str(cfg_mask_txt_dir / f"mask_{stem}.txt")
    )

    return df


dataset_inventory = add_segmentation_paths(dataset_inventory)
dataset_inventory_fake = add_segmentation_paths(dataset_inventory_fake)

/tmp/ipykernel_30252/4162983395.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it/datasets/output/ad/heatmap/result_878_sdv5_00027.png'
 '/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it/datasets/output/ad/heatmap/result_882_sdv5_00039.png'
 '/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it/datasets/output/ad/heatmap/result_903_sdv5_00009.png'
 '/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it/datasets/output/ad/heatmap/result_907_sdv5_00020.png'
 '/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it/datasets/output/ad/heatmap/result_918_sdv5_00039.png'
 '/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it/datasets/output/ad/heatmap/result_930_sdv5_00135.png'
 '/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it/datasets/output/ad

In [23]:
# Checkpoint save
dataset_inventory.to_csv(DATASET_CHECKPOINT_PATH)
dataset_inventory_fake.to_csv(FAKE_DATASET_CHECKPOINT_PATH)